# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a reproducible workflow for loading, exploring, and analyzing the FAIR² dataset using the `mlcroissant` library and pandas.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset Croissant metadata URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the Croissant dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata

# Display main overview
print(f"Dataset name: {metadata.name}")
print(f"Description: {metadata.description}")
print(f"Version: {getattr(metadata, 'version', 'N/A')}")

## 2. Data Overview
Review available record sets and their fields. All entities are referenced by their `@id` for precise identification.

In [ ]:
# List all RecordSets and their fields
if not hasattr(metadata, 'record_set'):
    print('No record sets found in metadata.')
else:
    record_sets = metadata.record_set
    print(f"Number of record sets: {len(record_sets)}\n")
    for rs in record_sets:
        print(f"RecordSet @id: {rs.id}")
        print(f"  Name: {getattr(rs, 'name', '(no name)')}")
        print(f"  Description: {getattr(rs, 'description', '(no description)')}")
        if hasattr(rs, 'field'):
            print(f"  Fields (@id):")
            for field in rs.field:
                print(f"    - {field.id} (label: {getattr(field, 'name', '')}, type: {getattr(field, 'data_type', '')})")
        print('---')

## 3. Data Extraction
Load data from all available record sets into DataFrames for analysis. Reference the record set and field `@id`s. If there is only one record set, it will be used as the main example.

In [ ]:
# List all record set @id's so we can use them
record_sets = getattr(metadata, 'record_set', [])
record_set_ids = [rs.id for rs in record_sets]
print('Record sets:', record_set_ids)

# Extract records into dataframes
dataframes = {}
for rs_id in record_set_ids:
    try:
        recs = list(dataset.records(record_set=rs_id))
    except Exception as e:
        print(f"Failed to load records for {rs_id}: {e}")
        continue
    if len(recs):
        df = pd.DataFrame(recs)
        dataframes[rs_id] = df
        print(f"{rs_id} loaded: {df.shape[0]} rows, {df.shape[1]} columns")
        print("Columns:", df.columns.tolist())
        print(df.head(2), "\n")

if not dataframes:
    print('No records were loaded into dataframes. Check dataset accessibility and Croissant schema definition.')
else:
    main_record_set_id = record_set_ids[0]  # use the first record set for further steps
    print(f"Proceeding with example record set @id: {main_record_set_id}")

## 4. Exploratory Data Analysis (EDA)
Apply basic filtering, normalization, and grouping examples. All operations use field `@id`, matching how mlcroissant exposes column names.

If you do not know the field IDs, refer to the columns printed in the previous section.

In [ ]:
if dataframes:
    df = dataframes[main_record_set_id]
    print("Available columns:", df.columns.tolist())
    # Try to determine a numeric field to demonstrate filtering/normalization
    example_numeric = None
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            example_numeric = col
            break
    if example_numeric is None:
        print('No numeric field found in this record set for EDA demo.')
    else:
        print(f"Using numeric field (by @id): {example_numeric}")
        threshold = df[example_numeric].quantile(0.75)
        filtered_df = df[df[example_numeric] > threshold].copy()
        print(f"Filtered records with {example_numeric} > {threshold} (75th percentile): {len(filtered_df)} rows.")

        # Normalize the chosen numeric field
        filtered_df[f"{example_numeric}_normalized"] = (
            filtered_df[example_numeric] - filtered_df[example_numeric].mean()
        ) / filtered_df[example_numeric].std(ddof=0)
        print(filtered_df[[example_numeric, f"{example_numeric}_normalized"]].head())

        # Try grouping by a likely categorical column
        possible_group_col = None
        for col in df.columns:
            if col != example_numeric and df[col].dtype == object and df[col].nunique() < 10:
                possible_group_col = col
                break

        if possible_group_col:
            print(f"Grouping by: {possible_group_col}")
            grouped_df = filtered_df.groupby(possible_group_col)[example_numeric].mean()
            print(grouped_df.head())
        else:
            print('No suitable group-by column found.')

## 5. Visualization
Visualize the numeric field distribution and relationship with a group field if available.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if dataframes and example_numeric:
    df = dataframes[main_record_set_id]
    plt.figure(figsize=(7,4))
    sns.histplot(df[example_numeric].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {example_numeric}")
    plt.xlabel(example_numeric)
    plt.show()

    if 'possible_group_col' in locals() and possible_group_col:
        plt.figure(figsize=(8,4))
        sns.boxplot(x=possible_group_col, y=example_numeric, data=df)
        plt.xticks(rotation=25)
        plt.title(f'{example_numeric} grouped by {possible_group_col}')
        plt.show()

## 6. Conclusion
In this notebook, we loaded the FAIR² dataset's Croissant schema, explored its available record sets and fields, extracted the primary data table to a pandas DataFrame, and performed basic data analysis using only entity `@id` references. You can extend this workflow for domain-specific analysis, advanced visualization, or downstream ML tasks. If you encountered issues extracting records, check dataset accessibility and Croissant schema details.